# C2.5 · Supply-chain research

**Function C — Red Teaming and Security Research with AI → Security Research with AI**  ·  *Security of AI*

Builds on **[C2.4 · Data-layer research](https://spbreed.github.io/cyber-commons/lessons/C2.4.html)**.

| | |
|---|---|
| Tools used | Sigstore, in-toto, OWASP AIBOM |

## What this lesson is

**What it covers.** Sign a model artefact with Sigstore and detect a tampered adapter.

**Why a security engineer needs it.** Adapter and LoRA provenance, registry tampering, dependency confusion in agent ecosystems. The control it builds is: verify provenance; sign and attest artefacts.

This is a **control** lesson: it builds the mechanism, then breaks it, so you can see what the control is actually load-bearing for rather than taking the claim on trust.

## 1 · The hook

A model, a dataset, an adapter and an MCP server all arrive the way any dependency arrives — from someone else, usually unsigned, usually pinned to a tag that can move. Provenance questions produce real answers only when they are specific.

> **At CyberTravels.** The third-party MCP server, the model, the OCR library and the adapter all arrived from somebody else, usually unsigned, usually pinned to a tag that can move. R4.

## 2 · The framework

```
   what arrives from someone else

   model weights   signed? by whom? pinned to a digest or a tag?
   dataset         provenance? licence? contaminated with your eval?
   adapter         who built it, against which base, verified how?
   MCP server      whose process? whose tool descriptions in your context?

   a moving tag is not a pin
```

Supply-chain research for AI systems is the ordinary software problem plus two
artefacts that have no mature process at all.

The ordinary part transfers directly: typosquatting, unsigned packages, new
packages with no soak time. The signals that predict a bad dependency have not
changed.

The two new artefacts:

- **Model weights.** Sigstore and in-toto attestation are technically possible
  and rare in practice. There is no download-count equivalent — "popular
  checkpoint" is not provenance, and a fine-tune of a fine-tune has a lineage
  nobody records.
- **Prompt and tool packages.** MCP servers, agent skill bundles, prompt
  libraries. These run *inside* your agent with your agent's authority, and
  there is no signing convention for them at all.

The honest output of this lesson includes stating where no answer currently
exists, because a risk assessment that invents one is worse than a gap.

## 3 · The procedure, as a skill

The same connector is a review when a person loads it and a block when an agent holding production credentials does. The skill scores the ordinary signals first, then re-scores weighted by the authority the artefact will execute with.

### The skill — [`skills/research/agent-supply-chain-assessment/SKILL.md`](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/research/agent-supply-chain-assessment/SKILL.md)

```yaml
name: agent-supply-chain-assessment
description: >-
  Score new packages and MCP connectors for typosquatting and ordinary
  supply-chain signals, then re-score them weighted by the authority the agent
  that loads them runs with. Use when an agent installs its own dependencies or
  connects to a server somebody added last week.
allowed-tools: Read, Grep, Glob
```

# The same package is a different risk inside an agent

Supply-chain assessment for agents differs in one term: the artefact runs with
the agent's authority. A connector that trips three ordinary signals is a
review; the same connector loaded by an agent holding production credentials is
a block. Weighting by authority is what turns the ordinary assessment into the
right answer.

## When to use this

When an agent can install packages, when an MCP server is added, and at any
review of what a coding agent is allowed to pull.

## Procedure

**1 — Establish the known-good set.** The packages this project actually uses.
Typosquat detection is a comparison against something; without the set it is a
spell-check.

**2 — Score edit distance against known-good names.** A distance of one or two
from a popular package, with a recent first-publication date, is the classic
shape. Report the package it imitates, not just the score.

**3 — Apply the ordinary signals.** Age, maintainer count, download history,
whether it was published after the agent asked for it, install scripts.

**4 — Re-score weighted by agent authority.** What credentials are in the
environment the artefact will execute in, and what the agent can reach. This is
the step that moves a review to a block and it needs no new information.

**5 — Report the two verdicts side by side.** Unweighted and authority-weighted.
The difference is the argument for gating what agents may install, and it is
easier to make with both numbers present.

## Output contract

```json
{
  "known_good": ["str"],
  "artefacts": [{"name": "str", "kind": "package|mcp", "distance": 0, "imitates": "str|null",
                 "signals": ["str"], "verdict": "allow|review|block"}],
  "authority": {"credentials_present": ["str"], "reachable": ["str"]},
  "weighted": [{"name": "str", "verdict": "allow|review|block", "moved": true}]
}
```

## Failure modes

- **Typosquat detection with no known-good set.** Everything is close to
  something.
- **Assessing the package and not the environment.** The authority is the term
  that differs.
- **Treating an MCP connector as configuration.** It is code that runs with the
  agent.

In [ ]:
# The code is not in this notebook. It is this file in the repository:
#   https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/research/agent-supply-chain-assessment/scripts/agent_supply_chain_assessment.py
SCRIPT = "skills/research/agent-supply-chain-assessment/scripts/agent_supply_chain_assessment.py"
REPO = "https://github.com/spbreed/cyber-commons"
BRANCH = "claude/vulnbench-setup-scheduling-81aqov"

import glob, os, subprocess, sys

CLONE = "/kaggle/working/cyber-commons"
_root = next((r for r in (".", "..", "../..", CLONE)
              if os.path.isfile(os.path.join(r, SCRIPT))), None)

if _root is None:
    # --filter=blob:none --sparse fetches the tree without the history or the
    # notebooks; `sparse-checkout set skills` then materialises only what runs.
    _c = subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none",
                         "--sparse", "--branch", BRANCH, REPO, CLONE],
                        capture_output=True, text=True)
    if _c.returncode:
        raise SystemExit(
            "could not fetch the skills: " + _c.stderr.strip()[-300:] +
            "\nOn Kaggle this needs Internet on in the notebook settings, which "
            "needs a phone-verified account. Without one, attach the dataset "
            "cybercommons/cyber-commons-skills instead — it holds the same tree.")
    subprocess.run(["git", "-C", CLONE, "sparse-checkout", "set", "skills"],
                   capture_output=True, text=True)
    _root = CLONE

_out = subprocess.run([sys.executable, os.path.join(_root, SCRIPT)],
                      capture_output=True, text=True,
                      env=dict(os.environ,
                               PYTHONPATH=os.path.join(_root, "skills/_runtime"),
                               PYTHONHASHSEED="0"))
print(_out.stdout, end="")
if _out.returncode:
    raise SystemExit(_out.stderr.strip()[-2000:])

## What you just proved

The two legitimate packages are allowed or reviewed; both typosquats are blocked with the distance and the package they imitate. The MCP connector trips three ordinary signals and is escalated to block once agent authority is weighted in. The final assessments state explicitly which signals are unavailable for model weights and tool packages.

## Your turn

Add one question to your third-party assessment: "does this artefact execute with our agent's authority?" Anything answering yes should not be assessed on the same scale as a library.

---

**Next → [C2.6 · Benchmarks, reproducibility and the research harness](https://spbreed.github.io/cyber-commons/lessons/C2.6.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/C2.5.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/C2.5.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*